In [ ]:
#RDDs

In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark import SparkContext

In [3]:
sc = SparkContext.getOrCreate()

In [ ]:
orders_payments_rdd = sc.textFile("olist_order_payments_dataset.csv")
payments_header = orders_payments_rdd.first()
payments = orders_payments_rdd.filter(lambda row: row != payments_header)


In [5]:
orders_rdd = sc.textFile("olist_orders_dataset.csv")

orders_header = orders_rdd.first()

orders = orders_rdd.filter(lambda row: row != orders_header)


In [6]:
#View the first row
header = orders_payments_rdd.first()
print(header)

"order_id","payment_sequential","payment_type","payment_installments","payment_value"


In [7]:
#View the first row
header = orders_rdd.first()
print(header)


"order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"


In [8]:
#View the first rows
orders_rdd.take(12)


['"order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"',
 'e481f51cbdc54678b7cc49136f2d6af7,"9ef432eb6251297304e76186b10a928d",delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00',
 '"53cdb2fc8bc7dce0b6741e2150273451",b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00',
 '"47770eb9100c2d0c44946d9cf07ec65d","41ce2a54c0b03bf3443c3d931a367089",delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00',
 '"949d5b44dbf5de918fe9c16f97b45f8a",f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00',
 'ad21c59c0840e6cb83a9ceb5573f8159,"8ab97904e6daea8866dbdbc4fb7aad2c",delivered,2018

In [9]:
#QUESTION : Count the total number oof orders
total_orders = orders.count()
print(total_orders)


99441


In [10]:
#Create an RDD from the orders CSV file and count total orders
orders_rdd = sc.textFile("olist_orders_dataset.csv")

orders_header = orders_rdd.first()

orders = orders_rdd.filter(lambda row: row != orders_header)

total_orders = orders.count()

print(total_orders)



99441


In [11]:
#Count how many orders have status = "delivered"
import csv

orders_clean = orders.map(lambda row: next(csv.reader([row])))

delivered_count = orders_clean.filter(
    lambda row: row[2] == "delivered"
).count()

print(delivered_count)


96478


In [12]:
# Return the top 5 most expensive order payments
# Payments are in olist_order_payments_dataset.csv
payments_rdd = sc.textFile("olist_order_payments_dataset.csv")

payments_header = payments_rdd.first()

payments = payments_rdd.filter(lambda row: row != payments_header)

payments_clean = payments.map(lambda row: next(csv.reader([row])))

top5 = payments_clean.map(
    lambda row: float(row[4])
).top(5)

print(top5)



[13664.08, 7274.88, 6929.31, 6922.21, 6726.66]


In [13]:
# Given an RDD of payment values, filter payments greater than 500 and multiply them by 1.1 (simulate tax)
updated_payments = payments_clean.map(
    lambda row: float(row[4])
).filter(
    lambda payment: payment > 500
).map(
    lambda payment: payment * 1.1
)

print(updated_payments.take(5))



[595.1, 596.926, 623.0400000000001, 2229.876, 1103.0030000000002]


In [14]:
#Find total revenue using reduce()

total_revenue = payments_clean.map(
    lambda row: float(row[4])
).reduce(
    lambda x, y: x + y
)

print(total_revenue)



16008872.119999535


In [ ]:
##### DATA FRAMES

In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import countDistinct

In [ ]:
spark = SparkSession.builder.appName("customers").getOrCreate()

In [55]:
#read CSV files into DataFrame
customers_df = spark.read.csv("olist_customers_dataset.csv", header=True, inferSchema=True)
customers_df.show(5)
##
payments_df = spark.read.csv ("olist_order_payments_dataset.csv", header=True, inferSchema=True)
payments_df.show(5)
##
orders_df = spark.read.csv ("olist_orders_dataset.csv", header=True, inferSchema=True)
orders_df.show(5)
##
order_items_df = spark.read.csv("olist_order_items_dataset.csv", header=True, inferSchema=True)
order_items_df.show(5)


+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [14]:
customers_df.printSchema()
##
payments_df.printSchema()
##
orders_df.printSchema()
##
order_items_df

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [21]:
#16. Load the customers dataset into a DataFrame and count the total number of customers.
customers_df.count()

99441

In [24]:
#17.Find the number of distinct customer states. 
# Use customers_dataset.csv
customers_df.select("customer_state").distinct().count()


27

In [ ]:
#18. Show rows where payment_value contains NULL values. 

payments_df.filter("payment_value = NULL").show(30)


+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
+--------+------------------+------------+--------------------+-------------+



In [31]:
#19. Calculate total revenue grouped by customer_state.
from pyspark.sql.functions import sum

 #STEP 1:Join orders + payments
orders_payments_df = orders_df.join(
    payments_df,
    on="order_id",
    how="inner"
)

#STEP 2: Join with customers
complete_df = orders_payments_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

#STEP 3: Group and sum
complete_df.groupBy("customer_state").agg(sum("payment_value").alias("total_revenue")).show(5)


+--------------+------------------+
|customer_state|     total_revenue|
+--------------+------------------+
|            SC| 623086.4300000004|
|            RO| 60866.19999999999|
|            PI|108523.96999999999|
|            AM|27966.930000000008|
|            RR|          10064.62|
+--------------+------------------+
only showing top 5 rows


In [32]:
#20 Calculate average payment_value grouped by order_status.
from pyspark.sql.functions import avg

orders_payments_df = orders_df.join(
    payments_df,
    on="order_id",
    how="inner"
)

orders_payments_df.groupBy("order_status").agg(avg("payment_value").alias("avg_payment_value")).show()

+------------+------------------+
|order_status| avg_payment_value|
+------------+------------------+
|     shipped| 151.9845283018868|
|    canceled|215.74638554216858|
|    invoiced|212.73227692307688|
|     created|            137.62|
|   delivered|153.06742794473521|
| unavailable|194.88368258859788|
|  processing|217.53639498432602|
|    approved|            120.54|
+------------+------------------+



In [33]:
# 21. Filter orders where payment_value is greater than 1000.
payments_df.filter("payment_value > 1000").show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|886b114d034f4ac1d...|                 1| credit_card|                  10|      2027.16|
|62d9b911d7c56cf45...|                 1| credit_card|                  10|      1002.73|
|f86d7bc39aab05299...|                 1|     voucher|                   1|      1302.42|
|4ff8e28200e5a7a50...|                 1| credit_card|                  10|      2288.31|
|ce6d150fb29ada17d...|                 1| credit_card|                  10|      1586.47|
|e11fec6c25945565c...|                 1| credit_card|                  10|      1995.69|
|360787554f4182460...|                 1| credit_card|                   3|      1783.44|
|d1ea13fc1406243e6...|                 1| credit_card|                  10|      1432.92|
|e41e58617

In [34]:
# 22. Filter customers from a specific state (e.g., SP).
customers_df.filter('customer_state = "SP" ').show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [35]:
#23. Replace NULL payment values with 0.
payments_df = payments_df.na.fill({"payment_value": 0}).show()


+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [44]:
# 24. Add a new column called total_order_value = price + freight_value (from order_items).

order_items_df = spark.read.csv("olist_order_items_dataset.csv", header=True, inferSchema=True)
order_items_df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [47]:
from pyspark.sql.functions import col

order_items_df = order_items_df.withColumn(
    "total_order_value",
    col("price") + col("freight_value")
)

order_items_df.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "total_order_value"
).show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value| total_order_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|             72.19|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|            259.83|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|            216.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|             25.78|
|00042b26cf59d7ce6...|            

In [ ]:
##SPARK SQL
#25. Create a temporary view called orders from the orders DataFrame.
#Creating a TempView
orders_df.createOrReplaceTempView("orders")

In [51]:
spark.sql("SELECT * FROM orders").show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [ ]:
#26.Write a SQL query to show the top 5 customers by total spending.

payments_df.createOrReplaceTempView("payments")
customers_df.createOrReplaceTempView("customers")

In [61]:
spark.sql("""
SELECT 
    c.customer_id,
    SUM(p.payment_value) AS total_spent
FROM customers c
JOIN orders o 
    ON c.customer_id = o.customer_id
JOIN payments p 
    ON o.order_id = p.order_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 5
""").show()

+--------------------+-----------+
|         customer_id|total_spent|
+--------------------+-----------+
|1617b1357756262bf...|   13664.08|
|ec5b2ba62e5743423...|    7274.88|
|c6e2731c5b391845f...|    6929.31|
|f48d464a0baaea338...|    6922.21|
|3fd6777bbce08a352...|    6726.66|
+--------------------+-----------+



In [63]:
# 27. Write a SQL query to calculate total revenue per state.
spark.sql("""
SELECT 
    c.customer_state,
    SUM(p.payment_value) AS total_revenue
FROM customers c
JOIN orders o 
    ON c.customer_id = o.customer_id
JOIN payments p 
    ON o.order_id = p.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
""").show(10)

+--------------+------------------+
|customer_state|     total_revenue|
+--------------+------------------+
|            SP| 5998226.960000021|
|            RJ|        2144379.69|
|            MG|1872257.2599999995|
|            RS|  890898.540000001|
|            PR| 811156.3800000008|
|            SC| 623086.4300000004|
|            BA| 616645.8200000003|
|            DF|         355141.08|
|            GO| 350092.3099999999|
|            ES|325967.55000000016|
+--------------+------------------+
only showing top 10 rows


In [ ]:
#28. Use a window function to find the highest spending customer in each state.
spark.sql("""
SELECT customer_state, customer_id, total_spent
FROM (
    SELECT 
        c.customer_state,
        c.customer_id,
        SUM(p.payment_value) AS total_spent,
        RANK() OVER (
            PARTITION BY c.customer_state 
            ORDER BY SUM(p.payment_value) DESC
        ) AS rank
    FROM customers c
    JOIN orders o 
        ON c.customer_id = o.customer_id
    JOIN payments p 
        ON o.order_id = p.order_id
    GROUP BY c.customer_state, c.customer_id
) ranked
WHERE rank = 1
""").show()

+--------------+--------------------+-----------+
|customer_state|         customer_id|total_spent|
+--------------+--------------------+-----------+
|            AC|711fff4266b53bae9...|     1251.7|
|            AL|f4db56f354c71370b...|    2269.98|
|            AM|25dcca1d4dd5e5ae8...|    1853.75|
|            AP|72a4878a79935a45e...|    1482.42|
|            BA|0b16ce67087d02bf8...|    3358.24|
|            CE|19365ad6a18ccf1ba...|    2734.11|
|            DF|926b6a6fb8b6081e0...|    4194.76|
|            ES|ec5b2ba62e5743423...|    7274.88|
|            GO|e0a2412720e9ea4f2...|    4809.44|
|            MA|71901689c5f3e5adc...|    3195.73|
|            MG|05455dfa7cd02f13d...|    6081.54|
|            MS|c6e2731c5b391845f...|    6929.31|
|            MT|f7622098214b4634b...|    3242.84|
|            PA|3be2c536886b2ea46...|    4042.74|
|            PB|3d979689f636322c6...|    4681.78|
|            PE|19b32919fa1198aef...|    3792.59|
|            PI|b44a611059f1fdd8d...|    1989.75|
